# Sentiment Analysis Experiment Notebook

## Research-style exploration for IMDB and SST

This notebook is designed as a reproducible, research-oriented NLP workspace. It demonstrates the end-to-end sentiment analysis workflow, but keeps the primary manual practice sections for Naive Bayes and Decision Tree intentionally open-ended so that they can be completed by hand later.

The notebook emphasizes:
- dataset exploration on IMDB and SST
- preprocessing and feature engineering
- model workflow reasoning
- evaluation and visualization
- engineering constraints such as leakage, sparsity, and reproducibility

In [ ]:
from pathlib import Path
import sys
import ast
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    sns = None

PROJECT_ROOT = next(
    (
        candidate for candidate in [Path.cwd(), Path.cwd().parent]
        if (candidate / "data" / "raw").exists() and (candidate / "src").exists()
    ),
    Path.cwd().parent,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = PROJECT_ROOT / "data" / "raw"

from src.preprocessing.text_processor import TextProcessor
from src.feature_extraction.count_vectorizer import CountVectorizer
from src.feature_extraction.tfidf_vectorizer import TfidfTransformer
from src.utils.helper import train_test_split
from src.evaluation.accuracy import accuracy
from src.evaluation.confusion_matrix import confusion_matrix
from src.models.knn.knn import KNN
from src.models.svms.svm import SVM
from src.models.regression.logistic import LogisticRegression
from src.models.neural_network.neural_network import NeuralNetworkClassifier

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (12, 5)


def find_csv_file(folder: Path):
    candidates = sorted(folder.glob("*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No CSV file found in {folder}")
    return candidates[0]


def decode_text(value):
    if pd.isna(value):
        return ""
    text = str(value)
    if text.startswith(("b'", 'b"')):
        try:
            decoded = ast.literal_eval(text)
            if isinstance(decoded, bytes):
                return decoded.decode("utf-8", errors="ignore")
            return str(decoded)
        except Exception:
            pass
    return text


def infer_text_label_columns(df):
    lower_map = {column.lower(): column for column in df.columns}
    text_candidates = ["sentence", "review", "text", "content", "comment"]
    label_candidates = ["target", "label", "sentiment", "polarity", "class"]

    text_col = next((lower_map[candidate] for candidate in text_candidates if candidate in lower_map), df.columns[0])
    label_col = next((lower_map[candidate] for candidate in label_candidates if candidate in lower_map), df.columns[-1])
    return text_col, label_col


def load_sentiment_dataset(path, name, limit=None):
    df = pd.read_csv(path)
    text_col, label_col = infer_text_label_columns(df)
    work = df[[text_col, label_col]].copy()
    work.columns = ["text", "label"]
    work["text"] = work["text"].map(decode_text).astype(str)
    work["label"] = work["label"].copy()
    if limit is not None:
        work = work.head(limit)
    return {
        "name": name,
        "path": path,
        "raw": df,
        "data": work,
        "text_col": text_col,
        "label_col": label_col,
    }


def text_length_stats(text_series):
    lengths = text_series.fillna("").astype(str).map(lambda value: len(value.split()))
    return lengths


def summarize_dataset(bundle):
    data = bundle["data"]
    lengths = text_length_stats(data["text"])
    label_counts = data["label"].value_counts(dropna=False).sort_index()
    summary = pd.DataFrame({
        "dataset": [bundle["name"]],
        "rows": [len(data)],
        "unique_labels": [label_counts.shape[0]],
        "avg_tokens": [lengths.mean()],
        "median_tokens": [lengths.median()],
        "min_tokens": [lengths.min()],
        "max_tokens": [lengths.max()],
    })
    return summary, label_counts, lengths


imdb_path = find_csv_file(DATA_ROOT / "imdb")
sst_path = DATA_ROOT / "sst" / "train.csv"

imdb_bundle = load_sentiment_dataset(imdb_path, "IMDB")
sst_bundle = load_sentiment_dataset(sst_path, "SST")

imdb_summary, imdb_label_counts, imdb_lengths = summarize_dataset(imdb_bundle)
sst_summary, sst_label_counts, sst_lengths = summarize_dataset(sst_bundle)

summary_table = pd.concat([imdb_summary, sst_summary], ignore_index=True)
summary_table

## 1. Research Motivation

Sentiment analysis remains a useful research benchmark because it combines language understanding with high-dimensional sparse data. In the classical ML setting, the problem is especially instructive: even when the number of features is large, the signal is often carried by a relatively small subset of words, phrases, and negation patterns.

This notebook keeps that tension visible. The objective is not just to obtain a score, but to inspect the engineering decisions that make a text pipeline work: cleaning text consistently, fitting vectorizers only on the training split, managing sparsity, and understanding where overfitting enters a classical model.

The two datasets used here serve complementary roles:
- IMDB offers a larger binary sentiment corpus for model workflow checks.
- SST offers a smaller and more structured sentiment dataset for quick inspection and reproducible experiments.

The deliberate design choice in this notebook is to fully demonstrate the workflow structure while leaving the detailed Naive Bayes and Decision Tree experiment sections open for manual practice later.

## 2. Dataset Exploration

The first question in any NLP experiment is not which model to use, but what the corpus actually looks like. Before vectorizing text, we inspect the dataset shape, label balance, and document length distribution. This is the cheapest way to detect broken labels, truncated text, or sampling artifacts.

The code below loads both IMDB and SST from `data/raw/`, normalizes their text/label columns into a shared schema, and prepares summary tables for later analysis.

In [ ]:
print('IMDB raw columns:', list(imdb_bundle['raw'].columns))
print('SST raw columns:', list(sst_bundle['raw'].columns))

print('\nIMDB preview:')
display(imdb_bundle['raw'].head(3))

print('\nSST preview:')
display(sst_bundle['raw'].head(3))

print('\nNormalized summary table:')
display(summary_table)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Class distributions
axes[0, 0].bar(imdb_label_counts.index.astype(str), imdb_label_counts.values, color='#4472c4')
axes[0, 0].set_title('IMDB class distribution')
axes[0, 0].set_xlabel('label')
axes[0, 0].set_ylabel('count')

axes[0, 1].bar(sst_label_counts.index.astype(str), sst_label_counts.values, color='#70ad47')
axes[0, 1].set_title('SST class distribution')
axes[0, 1].set_xlabel('label')
axes[0, 1].set_ylabel('count')

# Text length histograms
axes[1, 0].hist(imdb_lengths, bins=40, color='#5b9bd5', alpha=0.85)
axes[1, 0].set_title('IMDB token-length histogram')
axes[1, 0].set_xlabel('tokens per document')
axes[1, 0].set_ylabel('documents')

axes[1, 1].hist(sst_lengths, bins=40, color='#ed7d31', alpha=0.85)
axes[1, 1].set_title('SST token-length histogram')
axes[1, 1].set_xlabel('tokens per document')
axes[1, 1].set_ylabel('documents')

plt.tight_layout()

## 3. NLP Preprocessing Pipeline

Text preprocessing is not a cosmetic step. It defines the representation that every downstream model sees. If preprocessing is inconsistent, then the model is learning from artifacts rather than from sentiment-bearing language.

In this notebook, preprocessing is kept intentionally explicit:
- lowercasing to reduce surface-form sparsity
- HTML and punctuation cleaning to remove markup noise
- stopword removal to test whether frequent function words help or hurt the task
- tokenization to expose the model to discrete lexical units
- optional stemming or lemmatization as a later experiment

The practical tradeoff is important: stronger preprocessing reduces vocabulary size, but can also remove subtle sentiment cues such as negation, emphasis, or domain-specific tokens.

In [ ]:
processor = TextProcessor(remove_stopwords=True, remove_numbers=True)
preprocess_samples = (
    imdb_bundle['data']['text'].dropna().sample(3, random_state=42).tolist()
    + sst_bundle['data']['text'].dropna().sample(3, random_state=42).tolist()
)

preprocess_rows = []
for sample in preprocess_samples:
    cleaned = processor.clean(sample)
    tokens = processor.process(sample)
    preprocess_rows.append({
        'raw_text': sample[:180],
        'cleaned_text': cleaned[:180],
        'tokens': ' '.join(tokens[:25]),
        'token_count': len(tokens),
    })

preprocess_demo = pd.DataFrame(preprocess_rows)
preprocess_demo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sample_frame = preprocess_demo.copy()
sample_frame['raw_len'] = sample_frame['raw_text'].map(lambda text: len(str(text).split()))
sample_frame['clean_len'] = sample_frame['cleaned_text'].map(lambda text: len(str(text).split()))

axes[0].barh(range(len(sample_frame)), sample_frame['raw_len'], color='#5b9bd5', label='raw')
axes[0].barh(range(len(sample_frame)), sample_frame['clean_len'], color='#70ad47', alpha=0.7, label='cleaned')
axes[0].set_title('Example token counts before/after cleaning')
axes[0].set_xlabel('tokens')
axes[0].set_ylabel('example index')
axes[0].legend()

if sns is not None:
    sns.boxplot(data=pd.DataFrame({'raw': sample_frame['raw_len'], 'cleaned': sample_frame['clean_len']}), ax=axes[1])
else:
    axes[1].boxplot([sample_frame['raw_len'], sample_frame['clean_len']], labels=['raw', 'cleaned'])
axes[1].set_title('Token-length change after preprocessing')
axes[1].set_ylabel('tokens')

plt.tight_layout()

## 4. Feature Extraction

Classical sentiment pipelines live or die by the quality of their sparse representation. Count vectors preserve frequency information directly, while TF-IDF attenuates globally common words and often improves the behavior of linear models.

This notebook uses both representations because they expose different engineering tradeoffs:
- Count features are easier to inspect and explain.
- TF-IDF is often more stable for linear separators.
- Both are sparse, which keeps the memory footprint manageable for text corpora.

The main question to keep in mind is not whether the vocabulary can be built, but whether the resulting feature space remains interpretable, reproducible, and efficient.

In [ ]:
feature_sample = imdb_bundle['data'].dropna(subset=['text', 'label']).sample(min(1200, len(imdb_bundle['data'])), random_state=42).copy()
feature_sample['processed_text'] = feature_sample['text'].map(lambda text: ' '.join(processor.process(text)))
feature_texts = feature_sample['processed_text'].tolist()
feature_labels = feature_sample['label'].tolist()

feature_train_text, feature_test_text, feature_train_y, feature_test_y = train_test_split(
    np.array(feature_texts),
    np.array(feature_labels),
    test_size=0.2,
    random_state=42,
    stratify=True,
)

vectorizer = CountVectorizer(lowercase=True, ngram_range=(1, 2), max_features=5000)
X_train_counts = vectorizer.fit_transform(feature_train_text)
X_test_counts = vectorizer.transform(feature_test_text)

tfidf = TfidfTransformer()
X_train_tfidf = tfidf.fit_transform(X_train_counts)
X_test_tfidf = tfidf.transform(X_test_counts)

vocab_size = len(vectorizer.vocabulary_)
train_sparsity = 1.0 - (X_train_counts.nnz / (X_train_counts.shape[0] * X_train_counts.shape[1]))
tfidf_norms = np.sqrt(np.asarray(X_train_tfidf.power(2).sum(axis=1)).ravel())

column_sums = np.asarray(X_train_counts.sum(axis=0)).ravel()
inv_vocab = {index: token for token, index in vectorizer.vocabulary_.items()}
top_term_indices = np.argsort(column_sums)[-20:][::-1]
top_terms = pd.DataFrame({
    'token': [inv_vocab[index] for index in top_term_indices],
    'count': column_sums[top_term_indices],
})

feature_summary = pd.DataFrame({
    'metric': ['vocab_size', 'train_shape_0', 'train_shape_1', 'nnz', 'sparsity', 'tfidf_row_norm_mean'],
    'value': [vocab_size, X_train_counts.shape[0], X_train_counts.shape[1], X_train_counts.nnz, train_sparsity, tfidf_norms.mean()],
})

feature_bundle = {
    'train_text': feature_train_text,
    'test_text': feature_test_text,
    'train_y': feature_train_y,
    'test_y': feature_test_y,
    'count_vectorizer': vectorizer,
    'tfidf_transformer': tfidf,
    'X_train_counts': X_train_counts,
    'X_test_counts': X_test_counts,
    'X_train_tfidf': X_train_tfidf,
    'X_test_tfidf': X_test_tfidf,
}

display(feature_summary)
display(top_terms)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(top_terms['token'].head(12), top_terms['count'].head(12), color='#4472c4')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title('Top frequent tokens from IMDB sample')
axes[0].set_ylabel('count')

axes[1].hist(np.asarray(X_train_counts.getnnz(axis=1)).ravel(), bins=30, color='#70ad47', alpha=0.85)
axes[1].set_title('Document-level non-zero counts (sparsity view)')
axes[1].set_xlabel('non-zero features per document')
axes[1].set_ylabel('documents')

plt.tight_layout()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(['Count', 'TF-IDF'], [X_train_counts.nnz, X_train_tfidf.nnz], color=['#5b9bd5', '#ed7d31'])
ax.set_title('Non-zero entries: count vs TF-IDF')
ax.set_ylabel('nnz')
plt.tight_layout()

## 5. Model Experimentation Overview

This project contains a broader family of from-scratch models, but this notebook is intentionally selective.

The two primary models for later manual practice are:
- Naive Bayes
- Decision Tree

For those two sections, this notebook only provides a skeleton, helper hooks, and TODO prompts. The purpose is pedagogical: the user should still complete the detailed experiment workflow manually.

To keep the notebook useful as a workflow demonstration, secondary models are used for the fully wired example experiments:
- KNN
- Logistic Regression
- SVM
- Neural Network

This keeps the notebook complete enough to study, while preserving space for hands-on implementation in the main models.

### Naive Bayes and Decision Tree: manual practice templates

The notebook intentionally leaves the detailed implementation workflow open for these two models.

Use this space to practice:
- fitting the model only on training data
- comparing sparse and dense behavior
- testing alternative vectorization settings
- documenting hyperparameter choices
- writing your own error analysis notes

Recommended exercise structure:
1. write the training function
2. run the model on a small subset
3. inspect the confusion matrix
4. record qualitative mistakes
5. iterate on the preprocessing and representation choices

In [ ]:
def run_naive_bayes_manual(*args, **kwargs):
    """Placeholder for a manual Naive Bayes experiment.

    TODO:
    - choose a text representation
    - fit only on training data
    - document the smoothing choice
    - inspect mistakes by class
    """
    return {
        'status': 'manual-practice',
        'next_step': 'Implement fit, predict, and evaluation cells below this helper when ready.',
        'available_inputs': list(feature_bundle.keys()),
    }


def run_decision_tree_manual(*args, **kwargs):
    """Placeholder for a manual Decision Tree experiment.

    TODO:
    - choose a text representation
    - fit only on training data
    - control tree depth and split criteria
    - document overfitting behavior
    """
    return {
        'status': 'manual-practice',
        'next_step': 'Implement tree fitting and regularization experiments below this helper when ready.',
        'available_inputs': list(feature_bundle.keys()),
    }


manual_practice_notes = {
    'naive_bayes': 'Leave the full workflow as an exercise: fit, predict, evaluate, inspect errors.',
    'decision_tree': 'Leave the full workflow as an exercise: fit, prune/limit depth, evaluate, inspect errors.',
}
manual_practice_setup = {
    'shared_inputs': list(feature_bundle.keys()),
    'recommended_order': [
        'Choose count or TF-IDF features',
        'Fit on the training split only',
        'Predict on the held-out test split',
        'Compute accuracy, precision, recall, F1, and confusion matrix',
        'Inspect a few failure cases manually',
    ],
}

display(pd.Series(manual_practice_notes, name='practice_scope'))
display(pd.Series(manual_practice_setup, name='manual_workflow_template'))

### Secondary model demonstration

To keep the notebook fully educational, the fully wired experiment block below uses secondary models. This gives a complete training/evaluation loop without consuming the manual practice reserved for Naive Bayes and Decision Tree.

The example focuses on smaller datasets and compact feature representations so the workflow stays readable and fast enough for notebook exploration.

In [ ]:
secondary_bundle = imdb_bundle['data'].copy().dropna(subset=['text', 'label'])
secondary_bundle['text'] = secondary_bundle['text'].astype(str)
secondary_bundle['processed_text'] = secondary_bundle['text'].map(lambda text: ' '.join(processor.process(text)))

# Keep the workflow small enough for notebook experimentation.
secondary_sample = secondary_bundle.sample(min(1200, len(secondary_bundle)), random_state=42)
secondary_labels = pd.factorize(secondary_sample['label'])[0]
secondary_texts = secondary_sample['processed_text'].tolist()

sec_train_text, sec_test_text, sec_train_y, sec_test_y = train_test_split(
    np.array(secondary_texts),
    np.array(secondary_labels),
    test_size=0.2,
    random_state=42,
    stratify=True,
)

sec_vectorizer = CountVectorizer(lowercase=True, ngram_range=(1, 2), max_features=3000)
sec_train_counts = sec_vectorizer.fit_transform(sec_train_text)
sec_test_counts = sec_vectorizer.transform(sec_test_text)

sec_tfidf = TfidfTransformer()
sec_train = sec_tfidf.fit_transform(sec_train_counts)
sec_test = sec_tfidf.transform(sec_test_counts)

secondary_models = {
    'KNN': KNN(k=5),
    'Logistic Regression': LogisticRegression(learning_rate=0.1, n_iters=300, threshold=0.5, verbose=False),
    'SVM': SVM(learning_rate=0.001, lambda_param=0.01, n_iters=300),
}

secondary_results = []
secondary_predictions = {}
for model_name, model in secondary_models.items():
    model.fit(sec_train, sec_train_y)
    pred = model.predict(sec_test)
    secondary_predictions[model_name] = pred
    secondary_results.append({
        'model': model_name,
        'accuracy': accuracy(sec_test_y, pred),
    })

secondary_results_df = pd.DataFrame(secondary_results).sort_values('accuracy', ascending=False)
secondary_results_df

In [ ]:
def compute_classification_metrics(y_true, y_pred):
    cm, classes = confusion_matrix(y_true, y_pred)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0).astype(float) - tp
    fn = cm.sum(axis=1).astype(float) - tp

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) != 0)

    return {
        'classes': classes,
        'cm': cm,
        'accuracy': accuracy(y_true, y_pred),
        'precision_macro': precision.mean(),
        'recall_macro': recall.mean(),
        'f1_macro': f1.mean(),
    }


secondary_metric_rows = []
secondary_metric_maps = {}
for model_name, predictions in secondary_predictions.items():
    metrics = compute_classification_metrics(sec_test_y, predictions)
    secondary_metric_maps[model_name] = metrics
    secondary_metric_rows.append({
        'model': model_name,
        'accuracy': metrics['accuracy'],
        'precision_macro': metrics['precision_macro'],
        'recall_macro': metrics['recall_macro'],
        'f1_macro': metrics['f1_macro'],
    })

secondary_metrics_df = pd.DataFrame(secondary_metric_rows).sort_values('f1_macro', ascending=False)
secondary_metrics_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Comparison chart
if sns is not None:
    sns.barplot(data=secondary_metrics_df, x='model', y='accuracy', ax=axes[0], palette='viridis')
else:
    axes[0].bar(secondary_metrics_df['model'], secondary_metrics_df['accuracy'], color='#5b9bd5')
axes[0].set_title('Secondary model accuracy comparison')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=25)

# Confusion matrix for the best secondary model
best_secondary_model = secondary_metrics_df.iloc[0]['model']
best_secondary_cm = secondary_metric_maps[best_secondary_model]['cm']
best_secondary_classes = secondary_metric_maps[best_secondary_model]['classes']

if sns is not None:
    sns.heatmap(best_secondary_cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], xticklabels=best_secondary_classes, yticklabels=best_secondary_classes)
else:
    axes[1].imshow(best_secondary_cm, cmap='Blues')
    axes[1].set_xticks(range(len(best_secondary_classes)))
    axes[1].set_yticks(range(len(best_secondary_classes)))
    axes[1].set_xticklabels(best_secondary_classes)
    axes[1].set_yticklabels(best_secondary_classes)
axes[1].set_title(f'Confusion matrix: {best_secondary_model}')
axes[1].set_xlabel('predicted')
axes[1].set_ylabel('true')

plt.tight_layout()

best_secondary_model, best_secondary_cm

## 6. Engineering Notes

A strong NLP notebook should do more than produce outputs. It should surface the engineering constraints that shape the outputs.

Key points to observe in this workflow:
- Fit/transform boundaries matter. The vectorizer is fitted only on the training split to avoid leakage.
- Sparse matrices reduce memory pressure, but some classical implementations still require dense conversion.
- Reproducibility depends on fixed random seeds, stable preprocessing, and deterministic sampling.
- Better accuracy is not always better modeling: overfitting can appear quickly in high-dimensional text spaces.

The notebook remains intentionally open-ended for Naive Bayes and Decision Tree so that you can practice the experiment design yourself rather than only reviewing the final results.